<font size="6" color='grey'><b>KI-Agenten. Verstehen. Anwenden. Gestalten.</b></font> </br>

<font size="5" color='grey'><b>MCP HuggingFace für MCP-Tools</b></font> </br>

---

In [ ]:
#@title 🔧 Umgebung einrichten{ display-mode: "form" }
!uv pip install --system -q git+https://github.com/ralf-42/Agenten.git#subdirectory=04_modul
!uv pip install --system -q fastmcp langchain-mcp-adapters huggingface_hub

import os
os.environ["LANGSMITH_TRACING"]  = "true"
os.environ["LANGSMITH_PROJECT"]  = "M30-MCP-HuggingFace"
os.environ["LANGSMITH_ENDPOINT"] = "https://eu.api.smith.langchain.com"

from genai_lib.utilities import (
    check_environment, get_ipinfo, setup_api_keys,
    mprint, mermaid, show_trace
)
setup_api_keys(['OPENAI_API_KEY', 'LANGSMITH_API_KEY', 'HF_TOKEN'], create_globals=False)
print()
check_environment()
print()
get_ipinfo()

# 1 | Übersicht
---


<p><font color='black' size="5">
@tool vs. MCP — wann welches Muster?
</font></p>



Agenten können Tools auf zwei grundlegend verschiedene Weisen nutzen:

| Kriterium | `@tool` (lokal) | MCP (remote) |
|-----------|----------------|-------------|
| **Laufzeit** | Im Jupyter-Kernel | Separater Prozess / Server |
| **Wiederverwendung** | Nur im aktuellen Notebook | Jede App, jedes LLM |
| **Sprache** | Nur Python | Python, Node.js, beliebig |
| **Skalierung** | Einfache Prototypen | Production-Services |
| **Sicherheit** | Direkter Code-Zugriff | Isolierter Prozess |
| **Protokoll** | LangChain intern | JSON-RPC 2.0 über HTTP |

**Faustregel:** `@tool` für schnelle Prototypen und Kurs-Demos — MCP wenn Tools wiederverwendbar, sprachunabhängig oder produktionsreif sein sollen.


# 2 | Model Context Protocol (MCP)
---

<p><font color='black' size="5">Was ist MCP?</font></p>

Ein Protokoll ist ein Regelwerk, das bestimmt, wie zwei Systeme miteinander kommunizieren. Protokolle regeln die Datenübertragung in Computernetzwerken, bei der Internetkommunikation und zwischen Softwaresystemen.

**Zum Beispiel:**

- **HTTP** (Hypertext Transfer Protocol): Ermöglicht Websites die Kommunikation mit Browsern.
- **TCP/IP**: Definiert, wie Datenpakete im Internet geroutet werden.
- **JSON-RPC**: Ein Protokoll, das den Datenaustausch im JSON-Format ermöglicht.

Das **Model Context Protocol (MCP)** ist ein offenes Protokoll, das es großen Sprachmodellen (LLMs) ermöglicht, sich auf standardisierte Weise in externe Datenquellen und Tools zu integrieren. Dieses von Anthropic entwickelte Protokoll macht es KI-Modellen leicht, nahtlos mit einer Vielzahl von Tools und Datenquellen zusammenzuarbeiten — **ohne dass für jede Quelle spezifische API-Integrationen erforderlich sind**.





<p><font color='black' size="5">Warum wurde MCP entwickelt?</font></p>

Die Notwendigkeit von MCP ergibt sich aus den Ineffizienzen aktueller KI-API-Interaktionen. Derzeit ist der Aufbau von KI-Agenten, die Daten aus verschiedenen Quellen abrufen, **fragmentiert, repetitiv und schwer zu skalieren**. Jedes Tool spricht seine eigene Sprache und erfordert individuelle Integrationen. MCP reduziert diese Komplexität und minimiert den Entwicklungsaufwand.

```
Agent  →  MCP Client  →  MCP Server  →  Externe Ressource (DB, API, Filesystem)
       ←              ←              ←  Ergebnis
```

> Einmal als MCP-Server implementiert, funktioniert ein Tool mit **jedem kompatiblen LLM-Client** — Claude, GPT, LangGraph, u.a.

**Wichtige Ressourcen:**
- [Anthropic MCP](https://www.anthropic.com/news/model-context-protocol) — Offizielle Ankündigung und Spezifikation
- [MCP Server Gallery](https://github.com/modelcontextprotocol/servers) — Verfügbare Open-Source-Server

MCP ist ein **standardisiertes Protokoll**, das LLMs mit externen Tools verbindet — wie ein **USB-Standard für AI**. Einmal implementiert, funktioniert ein MCP-Tool mit jedem kompatiblen LLM-Client.

**Vier Kernkomponenten:**

| Komponente | Aufgabe | Beispiele |
|------------|---------|-----------|
| **MCP Host** | Die Anwendung, die den Client enthält und den Agenten ausführt | Chat-App, IDE-Assistent, Jupyter Notebook |
| **MCP Client** | Protokoll-Komponente innerhalb des Hosts — verbindet sich zu Servern, lädt Tool-Definitionen | `MultiServerMCPClient` |
| **MCP Server** | Stellt Tools bereit, implementiert JSON-RPC 2.0 — sprachunabhängig | FastMCP (Python), Node.js, beliebig |
| **LLM / Agent** | Entscheidet welche Tools gebraucht werden, orchestriert die Aufrufe | LangGraph ReAct-Agent |

> Ein Host kann mehrere Clients enthalten, ein Client kann sich zu mehreren Servern verbinden.


**Transport-Optionen:**

| Transport | Beschreibung | Einsatz |
|-----------|-------------|---------|
| `streamable_http` | HTTP POST mit JSON-RPC 2.0 | **Web-Services, Notebooks** ✅ |
| `stdio` | Standard Input/Output | Lokale Prozesse (Claude Desktop) |
| `sse` | Server-Sent Events | Echtzeit-Streaming vom Server |
| `websocket` | Bidirektionale Verbindung | Hochperformante Anwendungen |

> Dieses Modul verwendet `streamable_http` — ideal für Jupyter und Web-Deployments.

**Ablauf aus Nutzerperspektive** (Ergänzung zum technischen Protokoll-Ablauf in Section 3):

```
User:   "Wie viele Kunden habe ich?"

  1  Host fragt MCP Server:     Welche Tools sind verfügbar?
  2  Server antwortet:          [sql_abfrage, tabellen_liste, ...]
  3  Host → LLM:                Frage + verfügbare Tools
  4  LLM → Host:                "Nutze sql_abfrage('SELECT COUNT(*) FROM kunden')"
  5  Host → MCP Server:         Tool-Aufruf mit Parametern
  6  Server → Datenbank:        SQL-Abfrage ausführen → Ergebnis zurück
  7  Server → Host → LLM:       Ergebnis ("42")
  8  LLM → User:                "Sie haben aktuell 42 Kunden."
```

Der Host verwaltet den gesamten Ablauf — das LLM entscheidet nur, *welches Tool* gebraucht wird.
Der MCP Server entscheidet, *wie* das Tool ausgeführt wird (DB, API oder lokaler Code).


<p><font color='black' size="5">
Lokaler vs. öffentlicher MCP-Server
</font></p>



MCP-Server können auf zwei grundlegend verschiedene Arten betrieben werden:

| Merkmal | Lokaler Server | HF Space Server (dieses Modul) |
|---------|---------------|--------------------------------|
| **Adresse** | `127.0.0.1:8001/mcp` | `https://ralf42-simple-mcp.hf.space/mcp` |
| **Erreichbarkeit** | Nur lokal im Jupyter-Kernel | Öffentlich, weltweit |
| **Lebensdauer** | Nur während der Notebook-Session | Permanent (schläft nach Inaktivität ein) |
| **Deployment** | `subprocess.Popen(...)` im Notebook | Einmalig auf HF Space deployen |
| **MCP-Protokoll** | ✅ identisch | ✅ identisch |
| **Transport** | `streamable_http` | `streamable_http` |

> **Das MCP-Protokoll ist in beiden Fällen identisch** — nur die Adresse ändert sich.
> `MultiServerMCPClient` verbindet sich genauso — nur mit einer anderen URL.

**Faustregel:** Lokaler Server für schnelle Prototypen und Kurs-Demos — HF Space wenn der Server dauerhaft und öffentlich zugänglich sein soll.

> ℹ️ **Hinweis:** Wer zuerst mit einem lokalen MCP-Server arbeiten möchte, kann das Modul M30_MCP_Local verwenden. Dieses Modul (M30_MCP_HuggingFace) ist aber vollständig eigenständig nutzbar.


<p><font color='black' size="5">
Klassische Verschlüsselungsverfahren als MCP-Demo
</font></p>



Der HF-Space-Server stellt drei historische Algorithmen als MCP-Tools bereit — ideal um das Tool-Selection-Verhalten des Agenten zu demonstrieren, da sie unterschiedliche Parameter und Konzepte haben.

| Tool | Algorithmus | Typ | Prinzip |
|------|-------------|-----|---------|
| `caesar` | Caesar-Cipher | **Substitution** | Buchstabe um festen `shift` verschieben — A+3 → D |
| `vigenere` | Vigenère-Cipher | **Polyalph. Substitution** | Shift variiert je Position — Schlüsselwort bestimmt die Abfolge |
| `scytale` | Scytale | **Transposition** | Buchstaben werden umgeordnet, nicht ersetzt — wie Text auf einen Zylinder gewickelt |

**Substitution vs. Transposition — der Kernunterschied:**

```
Caesar   (Substitution):   HELLO  →  KHOOR   (Shift 3 — Buchstaben ändern sich)
Scytale  (Transposition):  HELLO  →  HLEOL   (rails=3 — Positionen ändern sich)
```

**Warum ist Vigenère schwerer zu knacken als Caesar?**
Bei Caesar hat jedes 'E' immer dieselbe Verschlüsselung → Häufigkeitsanalyse möglich.
Bei Vigenère ändert sich der Shift je nach Position → dasselbe 'E' wird unterschiedlich verschlüsselt.

**Alle drei Tools haben ein `decrypt`-Flag:**
```python
caesar(text="KHOOR", shift=3, decrypt=True)    # → "HELLO"
vigenere(text="RIJVS", key="KEY", decrypt=True) # → "HELLO"
scytale(text="HLEOL", rails=3, decrypt=True)    # → "HELLO"
```

> ⚠️ Historisch interessant, aber **nicht für echte Verschlüsselung geeignet.**


In [ ]:
#@markdown   <p><font size="4" color='green'>  flowchart: MCP + HF Space Architektur</font> </br></p>

diagram = '''
%%{init: {'theme':'dark'}}%%
flowchart TD
    U(["👤 User"])
    A["LangGraph Agent\ngpt-4o-mini"]
    C["MCP Client\nMultiServerMCPClient"]
    S["FastMCP Server\nHF Space · simple_mcp\nhttps://ralf42-simple-mcp.hf.space/mcp"]

    T1["🔑 caesar\nSubstitution — fester Shift"]
    T2["🗝️ vigenere\nPolyalph. Substitution — Schlüsselwort"]
    T3["📜 scytale\nTransposition — Zylinder-Umordnung"]

    U -->|Anfrage| A
    A -->|Tool Call| C
    C -->|HTTP POST JSON-RPC\nstreamable_http| S
    S --> T1 & T2 & T3
    T1 & T2 & T3 -->|Ergebnis| S
    S -->|Tool Result| C
    C -->|ToolMessage| A
    A -->|Antwort| U

    style A  fill:#2E7D32,color:#fff
    style C  fill:#1565C0,color:#fff
    style S  fill:#37474F,color:#fff
    style U  fill:#E65100,color:#fff
    style T1 fill:#4A148C,color:#fff
    style T2 fill:#4A148C,color:#fff
    style T3 fill:#4A148C,color:#fff
'''
mermaid(diagram, width=700)

# 3 | HF-MCP-Server erstellen
---

Der FastMCP-Server auf HF Spaces stellt 3 Crypto-Tools bereit — jedes mit einem einheitlichen `decrypt`-Flag:

| Tool | Parameter | Aufgabe |
|------|-----------|---------|
| `caesar` | `text`, `shift` (1–25), `decrypt` | Caesar-Substitution — fester Buchstaben-Shift |
| `vigenere` | `text`, `key` (Schlüsselwort), `decrypt` | Vigenère — polyalphabetische Substitution |
| `scytale` | `text`, `rails` (≥ 2), `decrypt` | Scytale — Transpositions-Cipher, Zylinder-Prinzip |


> ℹ️ **HF Space URL:** `https://ralf42-simple-mcp.hf.space/mcp`
> **Transport:** `streamable_http` (identisch zu M30)
> HF Spaces schlafen nach Inaktivität ein — der erste Request weckt den Space auf (~30 s).

In [ ]:
#@markdown   <p><font size="4" color='green'>  🔌 HF Space aufwecken + Verbindung prüfen</font> </br></p>

# ------------------------------------------------------------
import requests, time, os

SPACE_ID   = "Ralf42/simple_mcp"
SPACE_URL  = "https://ralf42-simple-mcp.hf.space"
MCP_URL    = f"{SPACE_URL}/mcp"
API_URL    = f"https://huggingface.co/api/spaces/{SPACE_ID}"
HF_TOKEN   = os.getenv("HF_TOKEN", "").strip()        # .strip() entfernt \r\n
HEADERS    = {"Authorization": f"Bearer {HF_TOKEN}"} if HF_TOKEN else {}

mprint(f"### 🔌 HF Space MCP-Server\n---")
print(f"Space:    {SPACE_ID}")
print(f"URL:      {SPACE_URL}")
print(f"Endpoint: {MCP_URL}")
print()

# ── Schritt 1: Aktuellen Status prüfen ───────────────────────────────────────
r = requests.get(API_URL, headers=HEADERS, timeout=10)
if r.ok:
    stage = r.json().get("runtime", {}).get("stage", "UNKNOWN")
    print(f"Status: {stage}")
    if "ERROR" in stage:
        print("❌ Space im Fehlerstatus — bitte app.py im HF Space korrigieren und neu deployen")
        raise SystemExit
    elif stage == "RUNNING":
        print("✅ Space bereits aktiv")
    else:
        print("⏳ Wecke Space auf ...")
        try:
            requests.get(SPACE_URL, timeout=10)
        except Exception:
            pass

        for i in range(24):   # max. 120 s
            time.sleep(5)
            r2 = requests.get(API_URL, headers=HEADERS, timeout=5)
            if r2.ok:
                stage = r2.json().get("runtime", {}).get("stage", "?")
                print(f"  [{(i+1)*5:>3}s] {stage}")
                if stage == "RUNNING":
                    break
                if "ERROR" in stage:
                    print("❌ Space im Fehlerstatus — bitte Logs prüfen (Zelle unten)")
                    raise SystemExit
        else:
            print("⚠️ Timeout — Zelle erneut ausführen")

# ── Schritt 2: MCP-Endpoint testen ───────────────────────────────────────────
print()
try:
    r = requests.post(
        MCP_URL,
        json={"jsonrpc": "2.0", "method": "tools/list", "id": 1},
        headers={"Content-Type": "application/json"},
        timeout=15
    )
    if r.status_code in [200, 400, 406]:
        print(f"✅ MCP-Endpoint erreichbar — HTTP {r.status_code}")
    elif r.status_code == 404:
        print(f"❌ HTTP 404 — /mcp-Route nicht gefunden")
        print(f"   → app.py prüfen: app.mount('/mcp', mcp.http_app())")
    else:
        print(f"⚠️ HTTP {r.status_code}")
except Exception as e:
    print(f"❌ {e}")

In [ ]:
#@markdown   <p><font size="4" color='green'>  🩺 HF Space Status + Logs prüfen</font> </br></p>
import requests, os

HF_TOKEN   = os.getenv("HF_TOKEN", "").strip()        # .strip() entfernt \r\n
SPACE_ID   = "Ralf42/simple_mcp"
HEADERS    = {"Authorization": f"Bearer {HF_TOKEN}"} if HF_TOKEN else {}
BASE_URL   = f"https://huggingface.co/api/spaces/{SPACE_ID}"

# ── Space-Status ──────────────────────────────────────────────────────────────
r = requests.get(BASE_URL, headers=HEADERS, timeout=10)
if r.ok:
    info    = r.json()
    stage   = info.get("runtime", {}).get("stage", "UNKNOWN")
    sdk     = info.get("sdk", "?")
    print(f"📦 Space:  {SPACE_ID}")
    print(f"🔧 SDK:    {sdk}")
    print(f"🟢 Status: {stage}")
    if stage == "SLEEPING":
        print("   ⚠️  Space schläft — erster Request weckt ihn auf (~30 s)")
    elif stage == "RUNNING":
        print(f"   MCP-Endpoint: https://ralf42-simple-mcp.hf.space/mcp")
else:
    print(f"⚠️ Status-API: HTTP {r.status_code}")

# ── Run-Logs (letzte Zeilen) ──────────────────────────────────────────────────
print("\n--- Run-Logs (letzte Zeilen) ---")
try:
    with requests.get(f"{BASE_URL}/logs/run", headers=HEADERS, stream=True, timeout=5) as log_r:
        lines = []
        for raw in log_r.iter_lines():
            if raw:
                line = raw.decode("utf-8", errors="replace")
                if line.startswith("data:"):
                    import json as _json
                    try:
                        entry = _json.loads(line[5:].strip())
                        msg = entry.get("data", {}).get("message", "")
                        if msg:
                            lines.append(msg)
                    except Exception:
                        pass
            if len(lines) >= 15:
                break
        for l in lines[-10:]:
            print(f"  {l}")
except Exception as e:
    print(f"  (Logs: {e})")

print(f"\n📎 Build-Logs: curl -N -H 'Authorization: Bearer $HF_TOKEN' '{BASE_URL}/logs/build'")
print(f"📎 Run-Logs:   curl -N -H 'Authorization: Bearer $HF_TOKEN' '{BASE_URL}/logs/run'")

# 4 | LangGraph Agent mit HF-Tools
---

Der `MultiServerMCPClient` lädt die Tool-Definitionen vom HF Space. Der LangGraph ReAct-Agent wählt automatisch das passende Crypto-Tool — basierend auf den Docstrings der Tools.

**Wichtig:** HF Spaces schlafen nach Inaktivität ein (Free Tier). Beim ersten Request kann es 10–30 Sekunden dauern, bis der Space aufgewacht ist. Folge-Requests sind sofort schnell.

In [ ]:
#@markdown   <p><font size="4" color='green'>  sequenceDiagram: Handshake + Tool-Aufruf</font> </br></p>

diagram = '''
sequenceDiagram
    autonumber
    actor User
    participant Agent as LangGraph Agent
    participant Client as MCP Client
    participant Server as FastMCP Server (SSE)

    Note over Client,Server: Phase 1 — Verbindung & Discovery
    Client->>Server: GET /sse (Establish Connection)
    Server-->>Client: SSE Stream Open (Endpoint URL)
    Client->>Server: HTTP POST JSON-RPC (initialize)
    Server-->>Client: SSE: Result (Capabilities)
    Client->>Server: HTTP POST JSON-RPC (tools/list)
    Server-->>Client: SSE: Result [caesar, vigenere, scytale]

    Note over Agent,Server: Phase 2 — Tool Execution
    User->>Agent: "Verschlüssele HALLO..."
    Agent->>Client: Call scytale(text="HALLO", rails=3)
    Client->>Server: HTTP POST JSON-RPC (tools/call)
    Server-->>Client: SSE: Result "HLEOLX"
    Client-->>Agent: ToolMessage("HLEOLX")
    Agent->>User: "Das Ergebnis lautet..."
'''
mermaid(diagram, width=950)

<p><font color='black' size="5">
MCP Client + Tools laden
</font></p>


In [ ]:
from langchain_mcp_adapters.client import MultiServerMCPClient
from langchain_core.messages import HumanMessage

client_crypto = MultiServerMCPClient({
    "crypto": {
        "transport": "streamable_http",
        "url": "https://ralf42-simple-mcp.hf.space/mcp"
    }
})

crypto_tools = await client_crypto.get_tools()

print(f"✅ {len(crypto_tools)} Crypto-Tools geladen:")
for t in crypto_tools:
    mprint(f"###   • {t.name}")
    print(f"     {t.description[:180]}")

<p><font color='black' size="5">
LangGraph ReAct-Agent mit Crypto-Tools
</font></p>

In [ ]:
from langchain.chat_models import init_chat_model
from langgraph.prebuilt import create_react_agent
from IPython.display import Image as IPImage, display

# gpt-5.1: bestes Tool-Following — kein temperature-Parameter erlaubt
llm = init_chat_model("openai:gpt-5.1")

crypto_agent = create_react_agent(
    llm,
    tools=crypto_tools,
    prompt=(
        "Du bist ein Experte für klassische Verschlüsselungsverfahren. "
        "Du hast Zugriff auf genau drei Tools über MCP.\n\n"

        "TOOL-AUSWAHL — wähle anhand dieser Signale:\n\n"

        "→ caesar   : Buchstabenverschiebung | fester Shift | ROT | 'um X Positionen'\n"
        "             Parameter: shift (Zahl 1-25)\n\n"

        "→ vigenere : Schlüsselwort | Passwort | Keyword | 'mit Wort XYZ'\n"
        "             Parameter: key (Wort, z.B. 'MCP', 'GEHEIM')\n"
        "             WICHTIG: Immer wenn ein Wort als Schlüssel genannt wird → vigenere!\n\n"

        "→ scytale  : Stabverfahren | Transposition | Spalten | Rails\n"
        "             Parameter: columns (Zahl)\n\n"

        "REGELN:\n"
        "- Rufe für JEDEN Schritt ein Tool auf — niemals selbst berechnen.\n"
        "- KRITISCH: Das Tool-Ergebnis aus Schritt N MUSS exakt als text-Parameter in Schritt N+1 übergeben werden.\n"
        "  Beispiel: Schritt 1 liefert 'KHOOR' → Schritt 2 erhält text='KHOOR', NICHT den Originaltext.\n"
        "- Antworte auf Deutsch und zeige alle Zwischenergebnisse."
    ),
)

In [ ]:
display(IPImage(crypto_agent.get_graph(xray=True).draw_mermaid_png()))

<p><font color='black' size="5">
Demo 1 — Caesar-Verschiebechiffre
</font></p>

In [ ]:
mprint("### 🔤 Demo 1 — Caesar-Verschiebechiffre\n---")

anfragen = [
    "Ich möchte 'Hallo Welt' geheim verschicken. "
    "Verschiebe jeden Buchstaben um 13 Positionen im Alphabet. "
    "Nutze dafür das passende Tool.",

    "Mein Freund hat mir 'KHOOR' geschickt und gesagt, "
    "er habe jeden Buchstaben um 3 Positionen nach vorne verschoben. "
    "Nutze das passende Tool und sag mir, was in der Nachricht steht.",

    "Zeige mir wie eine Verschiebechiffre funktioniert: "
    "Verschlüssele 'KI-Agenten sind mächtig' mit Shift 7. "
    "Nutze dann das verschlüsselte Ergebnis als Eingabe und stelle den Originaltext wieder her. "
    "Führe beide Schritte mit den verfügbaren Tools aus.",
]

for anfrage in anfragen:
    result = await crypto_agent.ainvoke({"messages": [HumanMessage(anfrage)]})
    mprint(f"**Anfrage:** {anfrage}\n\n**Antwort:** {result['messages'][-1].content}\n\n---")

In [ ]:
#@markdown   <p><font size="4" color='green'>  LangSmith Trace-Analyse</font> </br></p>

import time as _t; _t.sleep(2)
show_trace("M30-MCP-HuggingFace", limit=3, show_steps=True)

<p><font color='black' size="5">
Demo 2 — Vigenère-Schlüsselwort-Chiffre
</font></p>

In [ ]:
anfragen_vig = [
    "Verschlüssele 'HALLO WELT' mit dem Passwort 'KEY'. "
    "Nutze das passende Tool und zeige das Ergebnis.",

    "Ich habe den Text 'RIJVS' erhalten. "
    "Das vereinbarte Passwort lautet 'KEY'. "
    "Nutze das passende Tool und sag mir, was die ursprüngliche Botschaft ist.",

    "Demonstriere die Schlüsselwort-Methode: "
    "Verschlüssele 'ANGRIFF UM MITTERNACHT' mit dem Passwort 'GEHEIM'. "
    "Nutze dann das verschlüsselte Ergebnis als Eingabe und überprüfe durch Entschlüsselung, "
    "dass der Originaltext wiederhergestellt wird. "
    "Führe beide Schritte mit den verfügbaren Tools aus.",
]

mprint("### 🔑 Demo 2 — Vigenère-Schlüsselwort-Chiffre\n---")
for anfrage in anfragen_vig:
    result = await crypto_agent.ainvoke({"messages": [HumanMessage(anfrage)]})
    mprint(f"**Anfrage:** {anfrage}\n\n**Antwort:** {result['messages'][-1].content}\n\n---")

In [ ]:
#@markdown   <p><font size="4" color='green'>  LangSmith Trace-Analyse</font> </br></p>

import time as _t; _t.sleep(2)
show_trace("M30-MCP-HuggingFace", limit=3, show_steps=True)

<p><font color='black' size="5">
Demo 3 — Skytale-Transpositionschiffre
</font></p>

In [ ]:
anfragen_scy = [
    "Verschlüssele 'ATTACKATDAWN' mit dem antiken Stabverfahren — "
    "wickle den Text auf einen Stab mit 4 Spalten. "
    "Nutze das passende Tool.",

    "Der Text 'ACDTKATAWATN' wurde mit einem Stabverfahren (4 Spalten) erzeugt. "
    "Nutze das passende Tool und sag mir, was die ursprüngliche Botschaft ist.",

    "Demonstriere das Transpositionsverfahren: "
    "Verschlüssele 'GEHEIMEBOTSCHAFT' mit 4 Spalten. "
    "Nutze dann das verschlüsselte Ergebnis als Eingabe und stelle den Originaltext "
    "durch Entschlüsselung wieder her. "
    "Führe beide Schritte mit den verfügbaren Tools aus.",
]

mprint("### 📜 Demo 3 — Skytale-Transpositionschiffre\n---")
for anfrage in anfragen_scy:
    result = await crypto_agent.ainvoke({"messages": [HumanMessage(anfrage)]})
    mprint(f"**Anfrage:** {anfrage}\n\n**Antwort:** {result['messages'][-1].content}\n\n---")

In [ ]:
#@markdown   <p><font size="4" color='green'>  LangSmith Trace-Analyse</font> </br></p>

import time as _t; _t.sleep(2)
show_trace("M30-MCP-HuggingFace", limit=3, show_steps=True)


<p><font color='black' size="5">
Demo 4: Substitution vs. Transposition
</font></p>

In [ ]:
mprint("### 🔀 SubstitDemo 4: Substitution vs. Transpositionution vs. Transposition\n---")

# Agent demonstriert den Kernunterschied der drei Verfahren an einem Text
anfrage = (
    "Verschlüssele das Wort 'PYTHON' mit allen drei Verfahren: "
    "1) Caesar mit Shift 13, "
    "2) Vigenère mit Schlüssel 'MCP', "
    "3) Scytale mit rails=3. "
    "Erkläre danach den Unterschied zwischen Substitution (Caesar, Vigenère) "
    "und Transposition (Scytale)."
)
result = await crypto_agent.ainvoke({"messages": [HumanMessage(anfrage)]})
mprint(f"**Anfrage:** {anfrage}\n\n**Antwort:** {result['messages'][-1].content}")

In [ ]:
#@markdown   <p><font size="4" color='green'>  LangSmith Trace-Analyse</font> </br></p>

import time as _t; _t.sleep(2)
show_trace("M30-MCP-HuggingFace", limit=3, show_steps=True)

<p><font color='black' size="5">
Verbindung beenden
</font></p>

In [ ]:
mprint("### 🛑 MCP-Client schließen\n---")

# MultiServerMCPClient (>= 0.1.0) unterstützt kein Context-Manager-Pattern mehr
# → Referenz freigeben genügt; der HF Space läuft weiter
client_crypto = None
print("✅ MCP-Client-Referenz freigegeben")
print("ℹ️ Der HF Space MCP-Server läuft weiter — keine lokalen Prozesse zu beenden.")

# A | Aufgabe
---

<p><font color='darkblue' size="4">
📌 <b>Wichtig</b>
</font></p>

Die Aufgabestellungen unten bieten Anregungen, Sie können aber auch gerne eine andere Herausforderung angehen.

**Hinweis zur Lösungshilfe:**
> In diesem Kurs dürfen und sollen Sie generative KI auch als Unterstützung beim Lernen und Entwickeln nutzen. Wenn Sie bei einer Aufgabe festhängen, können Sie zum Beispiel Gemini in Google Colab verwenden, um Fehlermeldungen besser zu verstehen, Ideen für Teilschritte zu bekommen oder Code-Varianten zu prüfen.
> <br>**Wichtig ist nur:** Die KI dient als Lern- und Entwicklungshilfe. Der Schwerpunkt des Kurses bleibt darauf, KI-Agenten selbst zu verstehen, aufzubauen und gezielt weiterzuentwickeln.



<p><font color='black' size="5">Eigene Crypto-Tools und MCP-Experimente</font></p>



**Aufgabe 1 — Neues Tool: Atbash-Cipher**
Ergänze **deinen** HF Space `simple_mcp` um ein Tool `atbash(text: str) -> str`.
Atbash ersetzt jeden Buchstaben durch seinen gespiegelten Buchstaben (A↔Z, B↔Y, ...).
Teste mit: `"HELLO"` → `"SVOOL"`.
> Hinweis: Atbash ist eine spezielle Substitution — kein decrypt-Parameter nötig (ist selbst-invers wie ROT13).



**Aufgabe 2 — Caesar Brute-Force**
Füge in **deinen** Space ein Tool `caesar_bruteforce(text: str) -> str` hinzu, das alle 25 möglichen Shifts zurückgibt.
Lasse den Agenten einen unbekannt verschlüsselten Text entschlüsseln: `"KHOOR ZRUOG"`.
